# Thermodynamic Waddington Pipeline - tutorial

This notebook runs the whole pipeline on synthetic data (no download) and
explains each output. The same calls work on real data; the last cell shows the
AnnData entry point.

Install with `pip install -e ".[viz]"` for the plots.

In [ ]:
from thermodynamic_waddington import (
    make_synthetic_dataset, analyze, fit_landscape, FitConfig,
    developmental_coordinate, commitment_profile, committor_free_energy_profile,
    calibrate_fit, plots,
)

ds = make_synthetic_dataset(cells=200, genes=16, seed=7)
labels = list(ds.labels)
source, target = [sorted(set(labels))[0]], [sorted(set(labels))[-1]]
source, target

## One call

`analyze` runs everything and returns a compact report: the irreversibility
test, the landscape depth in kT, and (with source/target labels) the committor
commitment coordinate and barrier.

In [ ]:
report = analyze(ds.expression, ds.velocity, labels=labels,
                 source_labels=source, target_labels=target)
report.to_dict()

## The full fit and the committor

Drive the pipeline directly for the landscape object and the per-cell committor,
which is 0 at the progenitor and 1 at the terminal fate.

In [ ]:
cfg = FitConfig(neighbors=20, dimensions=6, seed=7)
fit = fit_landscape(ds.expression, ds.velocity, config=cfg, labels=labels)

q = developmental_coordinate(fit, source, target)
profile = commitment_profile(fit, source, target)
pmf = committor_free_energy_profile(q)

print('order:', profile.order)
print('commitment at:', profile.commitment_label)
print('barrier:', round(pmf.barrier_kt, 2), 'kT at q =', round(pmf.barrier_q, 2))

## Plots

One-line figures on any coordinate set (here the fit embedding; on real data pass
`adata.obsm['X_umap']`).

In [ ]:
plots.committor(q, coords=fit.embedding)
plots.free_energy_profile(q)
plots.landscape(fit.energies, coords=fit.embedding);

## On real data (AnnData)

With an AnnData that has a velocity layer (or spliced/unspliced to derive a
proxy), `analyze_adata` runs the pipeline and writes the committor and energy
back into `adata.obs`:

```python
from thermodynamic_waddington import analyze_adata
report = analyze_adata(adata, label_key='clusters',
                       source=['Ductal'], target=['Beta'])
adata.obs['tw_committor']   # written back
```